# Notebook 3: Knowledge Distillation Training
Output notebook ini berupa `bdd100k_cache.pkl`, `best_student_kd.pt`, `last_student_kd.pt`, dan CSV metrics.

In [ ]:
!pip install ultralytics==8.1.47
import ultralytics
ultralytics.checks()

In [ ]:
import os
import json
import glob
import pickle
from types import SimpleNamespace
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from ultralytics import YOLO
from ultralytics.utils.loss import v8DetectionLoss
from tqdm import tqdm
import random
import albumentations as A
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
import pandas as pd

In [ ]:
if not hasattr(torch, '_is_patched'):
    _original_load = torch.load

    def patched_load(*args, **kwargs):
        if 'weights_only' not in kwargs:
            kwargs['weights_only'] = False
        return _original_load(*args, **kwargs)

    torch.load = patched_load
    torch._is_patched = True

if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid

In [ ]:
BASE_PATH = "/kaggle/input/datasets/jimmyxwang/bdd100k"
 
SPLIT_DIRS = {
    "train": {
        "images": os.path.join(BASE_PATH, "train", "images"),
        "labels": os.path.join(BASE_PATH, "train", "labels"),
    },
    "val": {
        "images": os.path.join(BASE_PATH, "val", "images"),
        "labels": os.path.join(BASE_PATH, "val", "labels"),
    },
    "test": {
        "images": os.path.join(BASE_PATH, "test", "images"),
        "labels": os.path.join(BASE_PATH, "test", "labels"),
    },
}
 
VALID_CLASSES = sorted([
    "car", "truck", "bus", "person", "rider",
    "bike", "motor", "traffic light", "traffic sign"
])
CLASS_TO_ID = {name: idx for idx, name in enumerate(VALID_CLASSES)}
NUM_CLASSES  = len(CLASS_TO_ID)
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_NAME = "bdd100k_cache.pkl"

print(f"[INFO] Device  : {DEVICE}")
print(f"[INFO] Classes : {CLASS_TO_ID}")

def find_file_in_input(filename):
    """Mencari file di /kaggle/input secara otomatis jika diimpor dari output notebook lain"""
    for root, dirs, files in os.walk("/kaggle/input"):
        if filename in files:
            return os.path.join(root, filename)
    return None

def load_or_build_cache(time_filters=("daytime", "night")):
    direct_path = "/kaggle/input/datasets/yosephoktavianus/bdd100k-cache-from-notebook1-version-19/bdd100k_cache.pkl"
    if os.path.exists(direct_path):
        print(f"[INFO] Memuat cache anotasi secara langsung dari: {direct_path}")
        with open(direct_path, "rb") as f:
            return pickle.load(f)
            
    # 1. Jika direct path tidak ketemu => cek apakah ada di /kaggle/input (diimpor dari Notebook sebelumnya)
    input_path = find_file_in_input(CACHE_NAME)
    if input_path:
        print(f"[INFO] Memuat cache anotasi dari input: {input_path}")
        with open(input_path, "rb") as f:
            return pickle.load(f)
            
    # 2. Cek di /kaggle/working
    working_path = os.path.join("/kaggle/working", CACHE_NAME)
    if os.path.exists(working_path):
        print(f"[INFO] Memuat cache anotasi dari working: {working_path}")
        with open(working_path, "rb") as f:
            return pickle.load(f)
            
    # 3. Jika tidak ada, bangun dari nol (hanya terjadi di Notebook 1)
    print("[INFO] Cache tidak ditemukan. Membangun cache dari file JSON (Proses I/O)...")
    all_data = {}
    for split in ["train", "val", "test"]:
        labels_dir  = SPLIT_DIRS[split]["labels"]
        json_files  = glob.glob(os.path.join(labels_dir, "*.json"))
        result = {t: [] for t in time_filters}
        skipped = 0
        
        for jf in json_files:
            with open(jf) as f:
                data = json.load(f)
            tod = data.get("attributes", {}).get("timeofday", "")
            if tod in result:
                result[tod].append((jf, data))
            else:
                skipped += 1
        print(f"  [{split}] " + "  ".join(f"{t}={len(result[t])}" for t in time_filters) + f" (skipped={skipped})")
        all_data[split] = result
        
    # Simpan hasil ke working directory agar bisa didownload/dijadikan output dataset
    with open(working_path, "wb") as f:
        pickle.dump(all_data, f)
    print(f"[INFO] Cache berhasil disimpan di: {working_path}")
    return all_data

In [ ]:
class BDD100kDataset(Dataset):
    def __init__(self, data_list, images_dir, class_to_id, img_size=(640, 640), is_train=False):
        self.data_list   = data_list       
        self.images_dir  = images_dir
        self.class_to_id = class_to_id
        self.img_size    = img_size
        self.is_train    = is_train # Flag krusial agar val/test tidak teracak

        # ── Setup Pipeline Albumentations ──────────────────────────────
        # Format 'yolo' mengharapkan nilai [cx, cy, w, h] dalam skala 0.0 - 1.0
        # min_visibility=0.2 berarti jika gambar terpotong dan sisa objek < 20%, label dibuang
        if self.is_train:
            self.transform = A.Compose([
                # 1. Resize proporsional berdasarkan sisi terpanjang
                A.LongestMaxSize(max_size=max(self.img_size)),
                # 2. Tambahkan padding abu-abu agar menjadi kotak sempurna (640x640)
                A.PadIfNeeded(
                    min_height=self.img_size[1], 
                    min_width=self.img_size[0], 
                    border_mode=cv2.BORDER_CONSTANT, 
                    fill=(114, 114, 114) # Nilai abu-abu standar YOLO
                ),
                A.HorizontalFlip(p=0.5),
                A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
                A.Affine( # Pengganti ShiftScaleRotate
                    scale=(0.9, 1.1), 
                    translate_percent=(-0.0625, 0.0625), 
                    rotate=(-15, 15), 
                    p=0.4, 
                    fill=(114, 114, 114) 
                ),
                A.Blur(blur_limit=3, p=0.1)
            ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.2))
        else:
            self.transform = A.Compose([
                A.LongestMaxSize(max_size=max(self.img_size)),
                A.PadIfNeeded(
                    min_height=self.img_size[1], 
                    min_width=self.img_size[0], 
                    border_mode=cv2.BORDER_CONSTANT, 
                    fill=(114, 114, 114)
                )
            ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

    def __len__(self):
        return len(self.data_list)
 
    def __getitem__(self, idx):
        json_path, data = self.data_list[idx]
        img_name = data.get("name", os.path.basename(json_path).replace(".json", ""))
        img_path = os.path.join(self.images_dir, img_name + ".jpg")
 
        img = cv2.imread(img_path)
        if img is None:
            # Fallback jika gambar rusak/hilang
            return torch.zeros(3, *self.img_size), torch.zeros((0, 5))
 
        orig_h, orig_w = img.shape[:2]
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Albumentations bekerja paling baik di RGB
 
        objects = []
        frames  = data.get("frames", [])
        if frames:
            objects = frames[0].get("objects", [])
 
        bboxes = []
        class_labels = []

        for obj in objects:
            category = obj.get("category", "")
            if category not in self.class_to_id: continue                    
 
            box2d = obj.get("box2d")
            if box2d is None: continue                    
 
            x1, y1 = box2d["x1"], box2d["y1"]
            x2, y2 = box2d["x2"], box2d["y2"]
 
            x1, x2 = min(x1, x2), max(x1, x2)
            y1, y2 = min(y1, y2), max(y1, y2)
            if x2 <= x1 or y2 <= y1: continue
 
            # Kalkulasi format YOLO
            cx = np.clip((x1 + x2) / 2.0 / orig_w, 0, 1)
            cy = np.clip((y1 + y2) / 2.0 / orig_h, 0, 1)
            bw = np.clip((x2 - x1) / orig_w, 0.001, 1) # Hindari width 0
            bh = np.clip((y2 - y1) / orig_h, 0.001, 1) # Hindari height 0
 
            bboxes.append([cx, cy, bw, bh])
            class_labels.append(self.class_to_id[category])
 
        # ── Eksekusi Transformasi Albumentations ────────────────────────
        transformed = self.transform(image=img_rgb, bboxes=bboxes, class_labels=class_labels)
        
        img_transformed = transformed['image']
        bboxes_transformed = transformed['bboxes']
        labels_transformed = transformed['class_labels']

        # Konversi gambar ke Tensor PyTorch [Channels, Height, Width]
        img_tensor = torch.from_numpy(img_transformed).permute(2, 0, 1).float() / 255.0
 
        # Rangkai kembali format [class_id, cx, cy, w, h]
        final_labels = []
        for bbox, label in zip(bboxes_transformed, labels_transformed):
            final_labels.append([label, bbox[0], bbox[1], bbox[2], bbox[3]])

        labels_tensor = (torch.tensor(final_labels, dtype=torch.float32) if final_labels else torch.zeros((0, 5)))
        return img_tensor, labels_tensor

In [ ]:
def collate_fn(batch):
    imgs, labels = zip(*batch)
    imgs = torch.stack(imgs, 0)
 
    batch_labels = []
    for i, lbl in enumerate(labels):
        if len(lbl) > 0:
            bi = torch.full((len(lbl), 1), float(i))
            batch_labels.append(torch.cat([bi, lbl], dim=1))
 
    targets = (torch.cat(batch_labels, 0) if batch_labels else torch.zeros((0, 6)))
    return imgs, targets

def _subsample(data_list, max_samples, label=""):
    if max_samples is None or max_samples >= len(data_list):
        return data_list
    rng = random.Random(42)
    sampled = rng.sample(data_list, max_samples)
    print(f"    [{label}] subsample: {len(data_list)} → {max_samples}")
    return sampled

In [ ]:
def build_all_dataloaders(
    batch_size   = 16,
    img_size     = (640, 640),
    max_train_day    = None,   
    max_train_night    = None,   
    max_val_day  = None,   
    max_val_night= None,   
    max_test_day = None,   
    max_test_night= None,  
):
    all_data = load_or_build_cache(time_filters=("daytime", "night"))
 
    train_img = SPLIT_DIRS["train"]["images"]
    val_img   = SPLIT_DIRS["val"]["images"]
    test_img  = SPLIT_DIRS["test"]["images"]
 
    train_day_list = _subsample(all_data["train"]["daytime"], max_train_day, "train")
    train_night_list = _subsample(all_data["train"]["night"], max_train_night, "train")
    train_combined = train_day_list + train_night_list
    rng = random.Random(42)  
    rng.shuffle(train_combined)
    
    val_day_list   = _subsample(all_data["val"]["daytime"],   max_val_day,   "val_day")
    val_night_list = _subsample(all_data["val"]["night"],     max_val_night, "val_night")
    test_day_list  = _subsample(all_data["test"]["daytime"],  max_test_day,  "test_day")
    test_night_list= _subsample(all_data["test"]["night"],    max_test_night,"test_night")
 
    train_ds      = BDD100kDataset(train_combined,  train_img, CLASS_TO_ID, img_size, is_train=True)
    val_day_ds    = BDD100kDataset(val_day_list,    val_img,   CLASS_TO_ID, img_size, is_train=False)
    val_night_ds  = BDD100kDataset(val_night_list,  val_img,   CLASS_TO_ID, img_size, is_train=False)
    test_day_ds   = BDD100kDataset(test_day_list,   test_img,  CLASS_TO_ID, img_size, is_train=False)
    test_night_ds = BDD100kDataset(test_night_list, test_img,  CLASS_TO_ID, img_size, is_train=False)
 
    print("=" * 58)
    print(f"  train (day+night)  : {len(train_ds):6d}  ← FT teacher & KD")
    print(f"  val   daytime      : {len(val_day_ds):6d}  ← val per epoch")
    print(f"  val   night        : {len(val_night_ds):6d}  ← val per epoch")
    print(f"  test  daytime      : {len(test_day_ds):6d}  ← evaluasi final")
    print(f"  test  night        : {len(test_night_ds):6d}  ← evaluasi final")
    print("=" * 58 + "\n")
 
    common = dict(collate_fn=collate_fn, num_workers=4, pin_memory=True)
 
    return {
        "train"      : DataLoader(train_ds,      batch_size=batch_size, shuffle=True,  drop_last=True,  **common),
        "val_day"    : DataLoader(val_day_ds,    batch_size=batch_size, shuffle=False, drop_last=False, **common),
        "val_night"  : DataLoader(val_night_ds,  batch_size=batch_size, shuffle=False, drop_last=False, **common),
        "test_day"   : DataLoader(test_day_ds,   batch_size=batch_size, shuffle=False, drop_last=False, **common),
        "test_night" : DataLoader(test_night_ds, batch_size=batch_size, shuffle=False, drop_last=False, **common),
    }

In [ ]:
def make_loss_fn(model):
    loss_fn = v8DetectionLoss(model)
    hyp     = getattr(loss_fn, "hyp", {})
    if isinstance(hyp, dict) or not hasattr(hyp, "box"):
        hyp_d = hyp if isinstance(hyp, dict) else {}
        loss_fn.hyp = SimpleNamespace(
            box = hyp_d.get("box", 7.5),
            cls = hyp_d.get("cls", 0.5),
            dfl = hyp_d.get("dfl", 1.5),
        )
    return loss_fn

def compute_loss(loss_fn, out, imgs, targets):
    bs    = imgs.shape[0]
    batch = {
        "img"       : imgs,
        "batch_size": bs,
        "cls"       : targets[:, 1:2],
        "bboxes"    : targets[:, 2:],
        "batch_idx" : targets[:, 0:1],
    }
    result = loss_fn(out, batch) # out : tebakan model, batch : ground truth targets/labels
 
    if isinstance(result, (tuple, list)):
        loss_val = result[0]
        loss_det = result[1] if len(result) > 1 else result[0].detach()
    else:
        loss_val = result
        loss_det = result.detach()
 
    if loss_val.numel() > 1:
        loss_val = loss_val.sum()

    loss_val = loss_val / bs
 
    return loss_val, loss_det

In [ ]:
def xywh2xyxy(boxes): # mengubah format dari (cx,cy,w,h) menjadi (x1,y1,y2,2) atau (titik kiri atas, titik kanan bawah)
    out = boxes.clone()
    out[:, 0] = boxes[:, 0] - boxes[:, 2] / 2
    out[:, 1] = boxes[:, 1] - boxes[:, 3] / 2
    out[:, 2] = boxes[:, 0] + boxes[:, 2] / 2
    out[:, 3] = boxes[:, 1] + boxes[:, 3] / 2
    return out
 
def box_iou(box1, box2):
    inter_x1 = torch.max(box1[:, 0].unsqueeze(1), box2[:, 0].unsqueeze(0))
    inter_y1 = torch.max(box1[:, 1].unsqueeze(1), box2[:, 1].unsqueeze(0))
    inter_x2 = torch.min(box1[:, 2].unsqueeze(1), box2[:, 2].unsqueeze(0))
    inter_y2 = torch.min(box1[:, 3].unsqueeze(1), box2[:, 3].unsqueeze(0))
    inter_w = (inter_x2 - inter_x1).clamp(min=0)
    inter_h = (inter_y2 - inter_y1).clamp(min=0)
    inter   = inter_w * inter_h
    area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
    area2 = (box2[:, 2] - box2[:, 0]) * (box2[:, 3] - box2[:, 1])
    union = area1.unsqueeze(1) + area2.unsqueeze(0) - inter
    return inter / (union + 1e-7)

def decode_yolov8_output(raw_out, img_size=640, conf_thresh=0.25, iou_thresh=0.7, reg_max=16):
    combined = raw_out[0]                    
    B, C, N = combined.shape
    nc = NUM_CLASSES  # Jumlah kelas hasil adjust_yolov8_classes
 
    if C == 64 + nc: # Kondisi: 64 + nc Channels (Raw DFL, mis. 73 untuk nc=9 (mode train))
        box_raw, cls_raw = combined[:, :64, :], combined[:, 64:, :]   
        box_dist = F.softmax(box_raw.reshape(B, 4, reg_max, 8400), dim=2)               
        bins = torch.arange(reg_max, dtype=torch.float32, device=combined.device)    
        decoded = (box_dist * bins.view(1, 1, reg_max, 1)).sum(dim=2)  
 
        strides, grids_xy = [], []
        for s in [8, 16, 32]:
            gs = img_size // s
            y, x = torch.meshgrid(torch.arange(gs, dtype=torch.float32, device=combined.device), torch.arange(gs, dtype=torch.float32, device=combined.device), indexing="ij")
            grids_xy.append(torch.stack([x.flatten(), y.flatten()], dim=1))
            strides.extend([s] * (gs * gs))
 
        grid_xy = torch.cat(grids_xy, dim=0)                          
        strides = torch.tensor(strides, dtype=torch.float32, device=combined.device)                
 
        l, t, r, b = decoded[:, 0], decoded[:, 1], decoded[:, 2], decoded[:, 3]
        x1, y1 = (grid_xy[:, 0] - l) * strides / img_size, (grid_xy[:, 1] - t) * strides / img_size
        x2, y2 = (grid_xy[:, 0] + r) * strides / img_size, (grid_xy[:, 1] + b) * strides / img_size
        
        boxes = torch.stack([x1, y1, x2, y2], dim=2).clamp(0, 1)
        conf, cls_id = torch.sigmoid(cls_raw).permute(0, 2, 1).max(dim=2)                   
 
    elif C == 4 + nc:  # Kondisi: 14 Channels (Sudah di-decode oleh model.eval)
        boxes_raw = combined[:, :4, :]
        cls_probs = combined[:, 4:, :]
        
        # Output bawaan YOLOv8 eval berbentuk cx, cy, w, h dalam skala piksel asli (0-640)
        # Kita konversi menjadi x1, y1, x2, y2 ternormalisasi (0.0 - 1.0)
        cx = boxes_raw[:, 0, :] / img_size
        cy = boxes_raw[:, 1, :] / img_size
        w  = boxes_raw[:, 2, :] / img_size
        h  = boxes_raw[:, 3, :] / img_size
        
        x1 = cx - w / 2
        y1 = cy - h / 2
        x2 = cx + w / 2
        y2 = cy + h / 2
        
        boxes = torch.stack([x1, y1, x2, y2], dim=2).clamp(0, 1) # [B, 8400, 4]
        # Dalam mode eval, probabilitas kelas biasanya tidak perlu di-sigmoid lagi
        conf, cls_id = cls_probs.permute(0, 2, 1).max(dim=2)     # [B, 8400]
        
    else:
        raise ValueError(f"Bentuk tensor tidak dikenali! Jumlah Channel: {C}")
 
    # --- Blok Logika NMS Tetap Sama ---
    max_nms = 30000  # batas kandidat sebelum NMS (mirip Ultralytics max_nms), agar tidak lambat saat conf_thresh kecil
    results = []
    for i in range(B):
        mask = conf[i] > conf_thresh
        b_boxes, b_conf, b_cls = boxes[i][mask], conf[i][mask], cls_id[i][mask].float()              
 
        if len(b_boxes) == 0:
            results.append(torch.zeros((0, 6), device=combined.device))
            continue

        if len(b_boxes) > max_nms:
            top_idx = b_conf.argsort(descending=True)[:max_nms]
            b_boxes, b_conf, b_cls = b_boxes[top_idx], b_conf[top_idx], b_cls[top_idx]
 
        keep_idx = []
        for c in b_cls.unique():
            c_mask = b_cls == c
            c_boxes, c_scores = b_boxes[c_mask], b_conf[c_mask]
            order = c_scores.argsort(descending=True)
            kept = []
            while len(order) > 0:
                i_best = order[0].item()
                kept.append(c_mask.nonzero()[i_best].item())
                if len(order) == 1: break
                ious = box_iou(c_boxes[i_best:i_best+1], c_boxes[order[1:]])[0]
                order = order[1:][ious < iou_thresh]
            keep_idx.extend(kept)
 
        if not keep_idx:
            results.append(torch.zeros((0, 6), device=combined.device))
        else:
            results.append(torch.cat([b_boxes[keep_idx], b_conf[keep_idx].unsqueeze(1), b_cls[keep_idx].unsqueeze(1)], dim=1))
            
    return results

@torch.no_grad()
def compute_metrics(model, dataloader, nc=NUM_CLASSES, conf_thresh=0.001, iou_thresh=0.7, label="", img_size=640):
    """
    Menghitung precision, recall, F1, mAP50, mAP50-95 menggunakan
    ultralytics.utils.metrics.ap_per_class agar hasilnya identik
    secara numerik dengan model.val() bawaan Ultralytics.

    conf_thresh dibuat kecil (default 0.001, sesuai default Ultralytics val)
    karena ap_per_class membutuhkan kurva PR yang lengkap, bukan hasil
    pada satu titik confidence saja.
    """
    from ultralytics.utils.metrics import ap_per_class, box_iou as ul_box_iou

    model.eval()

    # 10 IoU thresholds: 0.50, 0.55, ..., 0.95 (sama seperti DetectionValidator.iouv)
    iouv = torch.linspace(0.5, 0.95, 10, device=DEVICE)
    niou = iouv.numel()

    all_tp   = []  # list of [N_det, 10] bool
    all_conf = []  # list of [N_det]
    all_pcls = []  # list of [N_det]
    all_tcls = []  # list of [N_gt]  -> target_cls untuk seluruh dataset

    for imgs, targets in tqdm(dataloader, desc=f"  Metrik [{label}]", leave=False):
        imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
        bs = imgs.shape[0]
        raw_out = model(imgs)
        # preds: list per-gambar, masing-masing [N,6] = (x1,y1,x2,y2,conf,cls), ternormalisasi 0-1
        preds = decode_yolov8_output(raw_out, img_size=img_size, conf_thresh=conf_thresh, iou_thresh=iou_thresh)

        for i in range(bs):
            det = preds[i]
            gt  = targets[targets[:, 0] == i]

            gt_cls = gt[:, 1].long() if len(gt) > 0 else torch.zeros((0,), dtype=torch.long, device=DEVICE)
            if len(gt_cls) > 0:
                all_tcls.append(gt_cls)

            if len(det) == 0:
                continue

            d_boxes = det[:, :4]
            d_conf  = det[:, 4]
            d_cls   = det[:, 5].long()

            if len(gt) == 0:
                # Semua deteksi adalah FP di semua IoU threshold
                tp = torch.zeros((len(det), niou), dtype=torch.bool, device=DEVICE)
            else:
                gt_boxes = xywh2xyxy(gt[:, 2:6])  # ground truth: cx,cy,w,h -> x1,y1,x2,y2 (0-1)

                iou = ul_box_iou(gt_boxes, d_boxes)               # [n_gt, n_det]
                correct_class = gt_cls[:, None] == d_cls[None, :]  # [n_gt, n_det]
                iou = iou * correct_class

                tp = torch.zeros((len(det), niou), dtype=torch.bool, device=DEVICE)
                iou_np = iou.cpu().numpy()
                for ti, thr in enumerate(iouv.cpu().tolist()):
                    matches = np.nonzero(iou_np >= thr)
                    matches = np.array(matches).T
                    if matches.shape[0]:
                        if matches.shape[0] > 1:
                            order = iou_np[matches[:, 0], matches[:, 1]].argsort()[::-1]
                            matches = matches[order]
                            matches = matches[np.unique(matches[:, 1], return_index=True)[1]]  # 1 det per gt
                            matches = matches[np.unique(matches[:, 0], return_index=True)[1]]  # 1 gt per det
                        tp[matches[:, 1].astype(int), ti] = True

            all_tp.append(tp)
            all_conf.append(d_conf)
            all_pcls.append(d_cls)

    if len(all_tp) == 0:
        tp_cat   = np.zeros((0, niou), dtype=bool)
        conf_cat = np.zeros((0,), dtype=np.float32)
        pcls_cat = np.zeros((0,), dtype=np.int64)
    else:
        tp_cat   = torch.cat(all_tp, dim=0).cpu().numpy()
        conf_cat = torch.cat(all_conf, dim=0).cpu().numpy()
        pcls_cat = torch.cat(all_pcls, dim=0).cpu().numpy()

    tcls_cat = (torch.cat(all_tcls, dim=0).cpu().numpy() if len(all_tcls) > 0
                else np.zeros((0,), dtype=np.int64))

    ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}
    names = {i: ID_TO_CLASS.get(i, str(i)) for i in range(nc)}

    if len(tp_cat) == 0 or len(tcls_cat) == 0:
        model.train()
        return {
            "precision": 0.0, "recall": 0.0, "f1": 0.0,
            "map50": 0.0, "map50_95": 0.0,
            "per_class": {names[c]: {"precision": 0.0, "recall": 0.0, "f1": 0.0, "ap50": 0.0, "ap50_95": 0.0} for c in range(nc)}
        }

    tp_res, fp_res, p_res, r_res, f1_res, ap_res, unique_classes, *_ = ap_per_class(
        tp_cat, conf_cat, pcls_cat, tcls_cat, names=names
    )

    # ap_res: [n_unique_classes, 10] -> AP per kelas per IoU threshold
    ap50    = ap_res[:, 0]          # AP@0.5
    ap50_95 = ap_res.mean(axis=1)   # AP@0.5:0.95

    map50    = float(ap50.mean())    if len(ap50)    > 0 else 0.0
    map50_95 = float(ap50_95.mean()) if len(ap50_95) > 0 else 0.0

    per_class_metrics = {}
    for idx, c in enumerate(unique_classes):
        per_class_metrics[names.get(int(c), str(c))] = {
            "precision": round(float(p_res[idx]), 4),
            "recall":    round(float(r_res[idx]), 4),
            "f1":        round(float(f1_res[idx]), 4),
            "ap50":      round(float(ap50[idx]), 4),
            "ap50_95":   round(float(ap50_95[idx]), 4),
        }
    # Kelas yang tidak punya GT sama sekali di split ini -> isi 0 agar key konsisten
    for c in range(nc):
        if names[c] not in per_class_metrics:
            per_class_metrics[names[c]] = {"precision": 0.0, "recall": 0.0, "f1": 0.0, "ap50": 0.0, "ap50_95": 0.0}

    mean_p  = float(p_res.mean())  if len(p_res)  > 0 else 0.0
    mean_r  = float(r_res.mean())  if len(r_res)  > 0 else 0.0
    mean_f1 = float(f1_res.mean()) if len(f1_res) > 0 else 0.0

    model.train()

    return {
        "precision": round(mean_p, 4),
        "recall":    round(mean_r, 4),
        "f1":        round(mean_f1, 4),
        "map50":     round(map50, 4),
        "map50_95":  round(map50_95, 4),
        "per_class": per_class_metrics
    }

@torch.no_grad()
def evaluate(model, dataloader, loss_fn, label=""):
    model.eval()
    total, n = 0.0, 0
    for imgs, targets in tqdm(dataloader, desc=f"  Eval [{label}]", leave=False):
        imgs    = imgs.to(DEVICE)
        targets = targets.to(DEVICE)
        out     = model(imgs)
        loss, _ = compute_loss(loss_fn, out, imgs, targets)
        total  += loss.item()
        n      += 1
    model.train()
    return total / max(n, 1)

class EarlyStopping:
    def __init__(self, patience=7, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = float("inf")
        self.counter    = 0
        self.triggered  = False
 
    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            print(f"  [EarlyStopping] No improvement {self.counter}/{self.patience} (best={self.best_loss:.4f}, current={val_loss:.4f})")
            if self.counter >= self.patience:
                self.triggered = True
                print(f"  [EarlyStopping] Triggered! Stopping after {self.patience} epochs.")
                return True
        return False

## ⚙️ KD v3 — Catatan Perubahan (Soft Weighting + Beta Schedule + CWD)

Tiga perubahan pada mekanisme distilasi (baseline & teacher **tidak** berubah, perbandingan tetap apple-to-apple):

1. **Soft confidence weighting** (`DetectionKD.forward`) — hard mask `t_conf > conf_thresh` **dihapus**. Semua anchor di-distill dengan bobot kontinu `w = sqrt(teacher_conf)` + weighted mean. Parameter `conf_thresh` tidak dipakai lagi. Target: memperbaiki precision-recall imbalance tanpa membuang informasi teacher.
2. **Beta cosine schedule** (`train_kd` loop) — `beta` tidak lagi konstan 0.25, melainkan cosine decay `0.35 → 0.10` sepanjang `total_epochs`. KD dominan di awal, ground truth dominan di akhir. Nilai beta per epoch tercatat di kolom `beta_kd` pada training history CSV.
3. **CWD feature distillation** (`ChannelWiseDistillation` menggantikan `MultiScaleFeatureDistillation`) — MSE diganti KL divergence pada softmax spasial per channel (τ=4), adapter 1×1 tanpa BN/ReLU. Bobot `GAMMA_FEAT = 0.03` (bukan 0.08 — skala loss berbeda; pantau kolom `feat` di progress bar epoch 1, target ~5–15% total loss).

⚠️ **Checkpoint lama (KD v1/Tuned) TIDAK kompatibel untuk resume** — struktur `feat_kd` dan optimizer berubah. Training KD v3 harus mulai dari epoch 1; resume hanya valid antar-checkpoint KD v3.


## 🧪 Ablasi: NO CWD (MSE Feature Distillation)

**Bagian dari ablation study KD v3** (3 orang, 1 teknik masing-masing). Notebook ini menonaktifkan **#3 CWD Feature Distillation** (dikembalikan ke versi pra-v3), sementara **#1 Soft Confidence Weighting, #2 Beta Cosine Schedule** tetap aktif persis seperti KD v3. Tujuannya: mengisolasi seberapa besar kontribusi teknik yang dinonaktifkan terhadap kenaikan skor (khususnya mAP50 Night).

- Checkpoint & CSV output notebook ini semuanya diberi suffix `_ablation_nocwd` supaya tidak tertukar dengan hasil KD v3 penuh atau ablation lain saat dikumpulkan kembali.
- **Kode lain di luar yang disebutkan di atas TIDAK diubah** dari notebook KD v3 (`output-notebooks-notebook3-kd.ipynb` / `notebook3-2-resume-kd.ipynb`).
- Baseline & teacher **tidak perlu di-retrain** — bandingkan langsung hasil run ini dengan KD v3 penuh dan Baseline yang sudah ada.


In [ ]:
import math

def adjust_yolov8_classes(pytorch_model, new_nc=9, img_size=640):
    """
    Mengubah classification head YOLOv8 dari 80 kelas ke new_nc
    beserta inisialisasi Focal Loss Prior untuk stabilisasi epoch awal.
    """
    detect_module = pytorch_model.model[-1] 
    
    # 1. UPDATE ATRIBUT INTERNAL DETECT MODULE DULU
    detect_module.nc = new_nc
    detect_module.no = new_nc + (detect_module.reg_max * 4)
    
    # 2. Update layer konvolusi dan inisialisasi bobot
    for i in range(detect_module.nl):
        old_conv = detect_module.cv3[i][-1] 
        in_channels = old_conv.in_channels
        
        new_conv = nn.Conv2d(
            in_channels, 
            new_nc, 
            kernel_size=1, 
            stride=1, 
            padding=0, 
            bias=True
        ).to(DEVICE)
        
        # Inisialisasi bobot (Weight) standar
        nn.init.normal_(new_conv.weight, mean=0.0, std=0.01)
        
        # --- INISIALISASI BIAS (FOCAL LOSS PRIOR) ---
        # Mengambil stride (lompatan piksel) pada scale saat ini (biasanya 8, 16, 32)
        s = float(detect_module.stride[i]) 
        
        # Menghitung bias logaritmik agar peluang awal deteksi objek sangat kecil (~1%)
        bias_value = math.log(5 / new_nc / (img_size / s) ** 2)
        
        # Terapkan nilai bias yang dihitung ke layer konvolusi yang baru
        nn.init.constant_(new_conv.bias, bias_value)
        
        # Ganti layer lama dengan layer baru
        detect_module.cv3[i][-1] = new_conv
        
    return pytorch_model

def setup_student(student_path="yolov8s.pt", new_nc=9):
    # 1. Load model mentah
    student = YOLO(student_path).model.to(DEVICE)
    
    # 2. Bedah dan sesuaikan Head ke 10 kelas
    student = adjust_yolov8_classes(student, new_nc=new_nc)
    
    student.train()
    
    # Full Fine-Tuning untuk student baseline
    for p in student.parameters():
        p.requires_grad_(True)
            
    return student

def load_or_create_teacher(teacher_path="yolov8l.pt", ckpt_path=None):
    teacher = YOLO(teacher_path).model.to(DEVICE)
    if ckpt_path is None:
        # Cari otomatis di input jika di-mount dari Notebook 1
        ckpt_path = find_file_in_input("best_teacher_finetuned_9class.pt")
        
    if ckpt_path and os.path.exists(ckpt_path):
        teacher = adjust_yolov8_classes(teacher, new_nc=NUM_CLASSES)
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        teacher.load_state_dict(ckpt["teacher_state"])
        print(f"[INFO] Teacher sukses dimuat dari: {ckpt_path}")
    else:
        print(f"[WARNING] Checkpoint teacher tidak ditemukan! Menggunakan Pretrained COCO.")
 
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    return teacher

def _extract_box_cls(raw_out):
    if isinstance(raw_out, tuple):
        raw_tensors = raw_out[1]  
    elif isinstance(raw_out, list):
        raw_tensors = raw_out     
    else:
        raise ValueError("Format output model YOLO tidak dikenali.")

    flattened = []
    for feat in raw_tensors:
        if isinstance(feat, torch.Tensor) and feat.dim() == 4:
            B, C, H, W = feat.shape
            flattened.append(feat.view(B, C, -1))
        elif isinstance(feat, (list, tuple)):
            for sub_feat in feat:
                if isinstance(sub_feat, torch.Tensor) and sub_feat.dim() == 4:
                    B, C, H, W = sub_feat.shape
                    flattened.append(sub_feat.view(B, C, -1))

    if len(flattened) == 0:
        raise RuntimeError("Gagal mengekstrak feature maps. List flattened kosong.")

    combined = torch.cat(flattened, dim=2)
    box = combined[:, :64, :]  
    cls = combined[:, 64:, :]  
    return box.permute(0, 2, 1), cls.permute(0, 2, 1)

class DetectionKD(nn.Module):
    def __init__(self, student_model, alpha=0.7, beta=0.35, temperature=4.0, temperature_dfl=2.0):
        # [KD v3] conf_thresh dihapus: hard mask diganti soft confidence weighting (lihat forward).
        # [KD v3] beta di sini hanya nilai AWAL -- di-update tiap epoch oleh beta schedule di train loop.
        super().__init__()
        self.task_loss_fn = make_loss_fn(student_model)
        self.alpha        = alpha
        self.beta         = beta
        self.T            = temperature        
        self.T_dfl        = temperature_dfl    
        self.reg_max      = 16                 
 
    def forward(self, images, student_raw, teacher_raw, targets):
        task_loss, _ = compute_loss(self.task_loss_fn, student_raw, images, targets)
        s_box, s_cls = _extract_box_cls(student_raw)   
        t_box, t_cls = _extract_box_cls(teacher_raw)
 
        # ============================================================
        # [KD v3 #1] SOFT CONFIDENCE WEIGHTING
        # Menggantikan hard mask (t_conf > conf_thresh) yang membuang
        # total anchor di bawah threshold. Sekarang SEMUA anchor ikut
        # di-distill dengan bobot kontinu w = sqrt(teacher_conf), lalu
        # dinormalisasi (weighted mean) agar skala loss tetap sebanding
        # dengan versi hard mask sebelumnya.
        # ============================================================
        t_conf = torch.sigmoid(t_cls.detach()).max(dim=-1).values      # (B, N) confidence teacher per anchor
        w      = t_conf.sqrt()                                         # sqrt -> angkat kontribusi anchor mid-confidence
        w_sum  = w.sum().clamp(min=1e-6)

        kl_cls = F.kl_div(
            F.log_softmax(s_cls / self.T, dim=-1),
            F.softmax(t_cls.detach() / self.T, dim=-1),
            reduction="none",
        ).sum(dim=-1)                                                  # (B, N) KL per anchor
        loss_cls_kd = (kl_cls * w).sum() / w_sum * (self.T ** 2)

        B, N, _ = s_box.shape
        s_dfl = s_box.reshape(B, N, 4, self.reg_max)
        t_dfl = t_box.reshape(B, N, 4, self.reg_max)
        kl_dfl = F.kl_div(
            F.log_softmax(s_dfl / self.T_dfl, dim=-1),
            F.softmax(t_dfl.detach() / self.T_dfl, dim=-1),
            reduction="none",
        ).sum(dim=-1).mean(dim=-1)                                     # (B, N) dirata-rata dari 4 sisi box
        loss_dfl_kd = (kl_dfl * w).sum() / w_sum * (self.T_dfl ** 2)
 
        total = self.alpha * task_loss + self.beta * (loss_cls_kd + loss_dfl_kd)
        return total, {
            "loss_det"    : task_loss.item(),
            "loss_cls_kd" : loss_cls_kd.item(),
            "loss_dfl_kd" : loss_dfl_kd.item(),
            "total"       : total.item(),
        }

class MultiScaleFeatureDistillation(nn.Module):
    """
    [Ablasi: NO CWD] Feature distillation versi MSE (sebelum KD v3).
    Menggantikan Channel-wise Distillation untuk mengisolasi kontribusi #3.
    Soft confidence weighting (#1) dan beta cosine schedule (#2) TETAP aktif.
    """
    def __init__(self, s_channels=[128, 256, 512], t_channels=[256, 512, 512]):
        super().__init__()
        self.adapters = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(sc, tc, kernel_size=1, bias=False),
                nn.BatchNorm2d(tc),
                nn.ReLU(inplace=True)
            ) for sc, tc in zip(s_channels, t_channels)
        ])

    def forward(self, student_feats, teacher_feats):
        loss = 0.0
        for i in range(len(student_feats)):
            adapted_s = self.adapters[i](student_feats[i])
            loss += F.mse_loss(adapted_s, teacher_feats[i].detach())
        return loss / len(student_feats)

In [ ]:
def train_kd(loaders, teacher_ckpt_path=None, epochs=22, total_epochs=50, lr=1e-4, patience=14, min_delta=1e-4, save_dir="/kaggle/working/kd_output"):
    os.makedirs(save_dir, exist_ok=True)
    teacher = load_or_create_teacher("yolov8l.pt", teacher_ckpt_path)
    student = setup_student("yolov8s.pt", new_nc=NUM_CLASSES)
 
    kd_module = DetectionKD(student, alpha=0.7, beta=0.35, temperature=4.0).to(DEVICE)  # soft weighting + beta schedule TIDAK diablasi di run ini (tetap aktif)
    feat_kd = MultiScaleFeatureDistillation().to(DEVICE)  # [Ablasi NO CWD] MSE feature distillation (versi pra-v3), bukan CWD

    # ================= [KD v3] HYPERPARAMETER =================
    # #2 Beta cosine schedule: beta bukan konstan lagi. Mulai tinggi
    #    (KD dominan -> transfer dark knowledge) lalu turun (ground
    #    truth dominan -> perbaiki recall di akhir training).
    BETA_START, BETA_END = 0.35, 0.10
    GAMMA_FEAT = 0.08
    # #3 [Ablasi NO CWD] MSE feature distillation -- gamma_feat dikembalikan ke skala
    #    MSE lama (0.08), BUKAN skala CWD (0.03).
    # ==========================================================

 
    teacher_feats, student_feats = {}, {}
    def get_hook(store, name):
        def hook(m, inp, out): store[name] = out
        return hook
 
    target_layers = [15, 18, 21] 
    for idx in target_layers:
        teacher.model[idx].register_forward_hook(get_hook(teacher_feats, f"feat_{idx}"))
        student.model[idx].register_forward_hook(get_hook(student_feats, f"feat_{idx}"))
 
    optimizer = torch.optim.AdamW(list(student.parameters()) + list(feat_kd.parameters()), lr=lr, weight_decay=0.05)
    
    # Implementasi LR Warmup & Cosine Annealing
    warmup_epochs = 3
    scheduler_warmup = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
    scheduler_cosine = CosineAnnealingLR(optimizer, T_max=(total_epochs - warmup_epochs))
    scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_cosine], milestones=[warmup_epochs])
    
    scaler = torch.amp.GradScaler('cuda')
 
    student_loss_fn = make_loss_fn(student)
    best_val_loss   = float("inf")
    ckpt_path       = os.path.join(save_dir, "best_student_kd_ablation_nocwd_9class.pt")
    last_ckpt_path  = os.path.join(save_dir, "last_student_kd_ablation_nocwd.pt")
    early_stop      = EarlyStopping(patience=patience, min_delta=min_delta)
 
    history = []
    per_class_history = [] 
    csv_path = os.path.join(save_dir, "student_kd_training_history_ablation_nocwd.csv")
    per_class_csv_path = os.path.join(save_dir, "student_kd_per_class_metrics_ablation_nocwd_9class.csv")

    print("\n" + "=" * 58)
    print("  FASE 2 [ABLASI NOCWD]: KNOWLEDGE DISTILLATION (YOLOv8s)")
    print("=" * 58)
 
    for epoch in range(1, epochs + 1):
        # [KD v3 #2] Cosine decay beta: BETA_START -> BETA_END sepanjang training
        progress = (epoch - 1) / max(total_epochs - 1, 1)
        kd_module.beta = BETA_END + 0.5 * (BETA_START - BETA_END) * (1 + math.cos(math.pi * progress))
        student.train()
        feat_kd.train()
        teacher_feats.clear()
        student_feats.clear()
 
        pbar = tqdm(loaders["train"], desc=f"Epoch {epoch}/{epochs} [KD]")
        epoch_loss, n_batch = 0.0, 0
 
        for imgs, targets in pbar:
            imgs    = imgs.to(DEVICE)
            targets = targets.to(DEVICE)
 
            optimizer.zero_grad()
            with torch.amp.autocast("cuda"):
                with torch.no_grad():
                    teacher_raw = teacher(imgs)
                student_raw = student(imgs)
                loss, ld = kd_module(imgs, student_raw, teacher_raw, targets)
 
                fl_val = 0.0
                if len(teacher_feats) == 3 and len(student_feats) == 3:
                    s_feat_list = [student_feats[f"feat_{idx}"] for idx in target_layers]
                    t_feat_list = [teacher_feats[f"feat_{idx}"] for idx in target_layers]
                    fl = GAMMA_FEAT * feat_kd(s_feat_list, t_feat_list)   # # [Ablasi NO CWD] MSE feature distillation (versi pra-v3), bukan CWD
                    loss += fl
                    fl_val = fl.item()
 
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(list(student.parameters()) + list(feat_kd.parameters()), 10.0)
            scaler.step(optimizer)
            scaler.update()
 
            epoch_loss += loss.item()
            n_batch += 1
            pbar.set_postfix({"loss": f"{loss.item()/imgs.shape[0]:.3f}", "det": f"{ld['loss_det']:.3f}", "kd": f"{ld['loss_cls_kd']:.3f}", "feat": f"{fl_val:.3f}"})
 
        # 1. Hitung Loss Validasi
        val_loss_day = evaluate(student, loaders["val_day"], student_loss_fn, "loss day")
        val_loss_night = evaluate(student, loaders["val_night"], student_loss_fn, "loss night")
        avg_val_loss = (val_loss_day + val_loss_night) / 2
        
        # 2. Hitung mAP & Metrics
        metrics_day = compute_metrics(student, loaders["val_day"], label="val day")
        metrics_night = compute_metrics(student, loaders["val_night"], label="val night")
        avg_map50 = (metrics_day["map50"] + metrics_night["map50"]) / 2
        avg_map50_95 = (metrics_day["map50_95"] + metrics_night["map50_95"]) / 2
        
        avg_train_loss = epoch_loss / max(n_batch, 1)

        # simpan metric per_class ke dalam csv
        for time_of_day, metrics_data in [("day", metrics_day), ("night", metrics_night)]:
            for class_name, cls_metrics in metrics_data["per_class"].items():
                per_class_history.append({
                    "epoch": epoch,
                    "time_of_day": time_of_day,
                    "class_name": class_name,
                    "precision": cls_metrics["precision"],
                    "recall": cls_metrics["recall"],
                    "f1": cls_metrics["f1"],
                    "ap50": cls_metrics["ap50"],
                    "ap50_95": cls_metrics["ap50_95"]
                })
        pd.DataFrame(per_class_history).to_csv(per_class_csv_path, index=False)

        # simpan metrics tiap epoch ke history.csv
        current_lr = optimizer.param_groups[0]["lr"]
        history.append({
            "epoch": epoch,
            "learning_rate": current_lr,
            "beta_kd": kd_module.beta,
            "train_loss": avg_train_loss,
            "val_day_loss": val_loss_day,
            "val_night_loss": val_loss_night,
            "avg_val_loss": avg_val_loss,
            "precision_day": metrics_day["precision"],
            "precision_night": metrics_night["precision"],
            "recall_day": metrics_day["recall"],
            "recall_night": metrics_night["recall"],
            "f1_day": metrics_day["f1"],
            "f1_night": metrics_night["f1"],
            "avg_f1": (metrics_day["f1"] + metrics_night["f1"]) / 2,
            "map50_day": metrics_day["map50"],
            "map50_night": metrics_night["map50"],
            "map5095_day": metrics_day["map50_95"],
            "map5095_night": metrics_night["map50_95"],
            "avg_map50": avg_map50,
            "avg_map5095": avg_map50_95
        })
        pd.DataFrame(history).to_csv(csv_path, index=False)
 
        print(
            f"  [KD Epoch {epoch:03d}]\n"
            f"  Train Loss={avg_train_loss:.4f} | beta_kd={kd_module.beta:.4f}\n"
            f"  Val-Day={val_loss_day:.4f} | Val-Night={val_loss_night:.4f} | Avg Val Loss={avg_val_loss:.4f}\n"
            f"  mAP50-Day={metrics_day['map50']:.4f} | mAP50-Night={metrics_night['map50']:.4f} | Avg mAP50={avg_map50:.4f}\n"
            f"  mAP50-95-Day={metrics_day['map50_95']:.4f} | mAP50-95-Night={metrics_night['map50_95']:.4f} | Avg mAP50-95={avg_map50_95:.4f}\n"
            f"  {'-'*58}"
        )
 
        # 3. Simpan jika LOSS membaik
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                "epoch": epoch,
                "student_state": student.state_dict(),
                "feat_kd_state": feat_kd.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "scaler_state": scaler.state_dict(),
                'val_day_loss' : val_loss_day,
                'val_night_loss': val_loss_night,
                "val_loss": avg_val_loss,
                "best_val_loss": best_val_loss,
                "val_map50": avg_map50,
                "val_map50_95": avg_map50_95,
                "metrics_day": metrics_day,
                "metrics_night": metrics_night,
                "early_stop_counter": early_stop.counter
            }, ckpt_path)
            print(f"  ✓ Best student KD model disimpan (Loss: {avg_val_loss:.4f} | mAP50: {avg_map50:.4f})")
 
        # Simpan last checkpoint di akhir setiap epoch
        torch.save({
            "epoch": epoch,
            "student_state": student.state_dict(),
            "feat_kd_state": feat_kd.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "scaler_state": scaler.state_dict(),
            'val_day_loss' : val_loss_day,
            'val_night_loss': val_loss_night,
            "val_loss": avg_val_loss,
            "best_val_loss": best_val_loss,
            "val_map50": avg_map50,
            "val_map50_95": avg_map50_95,
            "metrics_day": metrics_day,
            "metrics_night": metrics_night,
            "early_stop_counter": early_stop.counter
        }, last_ckpt_path)
        print(f"  ✓ Last student KD checkpoint disimpan ke: {last_ckpt_path}")
 
        if early_stop(avg_val_loss): break
        scheduler.step()
        
    return ckpt_path, last_ckpt_path

In [ ]:
if __name__ == "__main__":
    SAVE_DIR   = "/kaggle/working/kd_output"
    BATCH_SIZE = 16    

    BUILD_CACHE_ONLY = False
 
    if BUILD_CACHE_ONLY:
        print("[INFO] Mode BUILD_CACHE_ONLY aktif.")
        print("[INFO] Membangun cache dataset tanpa memuat DataLoader atau Model...")
        _ = load_or_build_cache(time_filters=("daytime", "night"))
        print("[INFO] Selesai! Silakan 'Save Version', lalu tambahkan output notebook ini ke Input.")
        
    else:
        loaders = build_all_dataloaders(
            batch_size    = BATCH_SIZE,
            max_train_day   = 10000,   
            max_train_night = 10000,  
            max_val_day   = 2000,
            max_val_night = 2000,
            max_test_day  = 2000,
            max_test_night= 2000,
        )
     
        student_ckpt, last_ckpt = train_kd(
            loaders    = loaders,
            teacher_ckpt_path = None,
            epochs     = 22,     
            total_epochs = 50,
            lr         = 1e-4,
            patience   = 14,      
            min_delta  = 1e-4,
            save_dir   = SAVE_DIR,
        )